# ASR Benchmark Analysis: Indian Conversational Telephony

This notebook analyzes the performance of three ASR models on a custom dataset of Bangalore-specific locality names spoken in a Hinglish conversational context.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate metrics from the benchmark
data = {
    'Model': ['Deepgram-nova-2', 'Groq-whisper-turbo', 'Whisper-large-v3'],
    'Avg WER': [0.81, 1.05, 1.06],
    'Entity Accuracy (%)': [58.8, 41.1, 58.8],
    'Avg Latency (s)': [0.67, 0.34, 1.84]
}

df = pd.DataFrame(data)
df

## 1. Latency Comparison
Groq demonstrates a significant lead in inference speed due to its LPU-accelerated infrastructure.

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))
colors = ['#4A90E2', '#50E3C2', '#F5A623']
ax = sns.barplot(x='Model', y='Avg Latency (s)', data=df, palette=colors)

plt.title('Inference Latency Comparison (Lower is Better)', fontsize=14, fontweight='bold')
plt.ylabel('Latency (seconds)')
plt.xlabel('')

for p in ax.patches:
    ax.annotate(format(p.get_height(), '.2f') + 's', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha = 'center', va = 'center', 
                xytext = (0, 9), 
                textcoords = 'offset points')

plt.tight_layout()
plt.show()

## 2. Accuracy vs. Latency Tradeoff
This scatter plot highlights the production "Sweet Spot". Deepgram and Whisper share the highest entity accuracy (thanks to the transliteration layer), but Deepgram is significantly faster than the local GPU Whisper.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Avg Latency (s)', y='Entity Accuracy (%)', hue='Model', size='Avg WER', 
                sizes=(100, 500), data=df, palette=colors, style='Model')

plt.title('Production Tradeoff: Accuracy vs. Latency', fontsize=14, fontweight='bold')
plt.xlabel('Latency (seconds) - Faster --->')
plt.ylabel('Entity Accuracy (%) - Better --->')
plt.grid(True, linestyle='--', alpha=0.7)

for i in range(df.shape[0]):
    plt.text(df['Avg Latency (s)'][i]+0.05, df['Entity Accuracy (%)'][i], df['Model'][i], fontsize=10)

plt.tight_layout()
plt.show()

## Key Findings
1. **Groq Speed**: Groq is **5.4x faster** than local GPU Whisper.
2. **Script Mismatch**: Without our transliteration layer, Whisper models would have 0% entity accuracy due to script mismatch.
3. **Production Choice**: **Deepgram Nova-2** remains the most robust choice for Indian telephony due to its balanced performance across script and noise.